# GNN Training and Serving Pipeline Test

This notebook demonstrates how to execute the GNN training pipeline and serving inference code entirely within a local notebook environment, hitting your actual Google Spanner database for data without deploying to Vertex AI or storing artifacts in Google Cloud Storage.

It uses the exact same code blocks found in:
- `gnn/tests/run_pipeline_local.py`
- `gnn/src/train_hetgnn.py`
- `gnn/src/serve.py`

In [ ]:
import os
import sys
import asyncio
import argparse
import datetime
from pathlib import Path
from unittest.mock import patch

# Ensure gnn/src is in the Python path for train_hetgnn, serve, and their utils.
src_path = str((Path.cwd().resolve().parent / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# run_pipeline_local lives alongside this notebook in gnn/tests/ — add explicitly
# so the import works even when the notebook is executed outside Jupyter
# (e.g. via nbconvert or a CI script that doesn't inject the notebook directory).
tests_path = str(Path.cwd().resolve())
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

import train_hetgnn
import serve
import run_pipeline_local

## Configuration

Define the parameters for Spanner connectivity and the time windows for training and test data.

- **Training data**: `TRAIN_FROM` → `TRAIN_TO` — snapshots used to fit the model.
- **Test data**: `TEST_FROM` → `TEST_TO` — held-out snapshots used to evaluate inference.

Leave the time-window variables commented out to fetch the most recent N snapshots from Spanner.
Uncomment and set specific datetimes to reproduce a historical training/test split.

In [ ]:
# Credential auth: check for networkagent.json in ../src (local dev), fall back to ADC.
# Mirrors the approach used in gnn/src/utils/data.py (SpannerDataset.__init__).
_local_creds = str((Path.cwd().resolve().parent / "src" / "networkagent.json").resolve())
creds_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", _local_creds)
if creds_path and os.path.exists(creds_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_path
    print(f"Using service-account credentials: {creds_path}")
else:
    print("networkagent.json not found — will use Application Default Credentials (ADC)")

project_id = os.getenv("GOOGLE_CLOUD_PROJECT", "agents-1234")
spanner_instance = os.getenv("SPANNER_INSTANCE", "networktopology-instance")
spanner_database = os.getenv("SPANNER_DATABASE", "networktopology-db")
interval_minutes = 0.5  # rca.md: 30 sec snapshot cadence
num_snapshots = 100

# ── Time windows (optional — uncomment to use a fixed historical split) ────────
# Training data: April 14 18:00 → 19:45  (21 snapshots at 5-min intervals)
# TRAIN_FROM = datetime.datetime(2026, 4, 14, 18,  0, 0)
# TRAIN_TO   = datetime.datetime(2026, 4, 14, 19, 45, 0)

# Test / held-out data: April 14 19:45 → 20:00  (4 snapshots at 5-min intervals)
# TEST_FROM  = datetime.datetime(2026, 4, 14, 19, 45, 0)
# TEST_TO    = datetime.datetime(2026, 4, 14, 20,  0, 0)

# ── Args namespaces (shared Spanner config + per-window time bounds) ──────────
def _base_args(**kwargs):
    return argparse.Namespace(
        project=project_id,
        spanner_instance=spanner_instance,
        spanner_database=spanner_database,
        interval_minutes=interval_minutes,
        num_snapshots=num_snapshots,
        **kwargs,
    )

# Build training args — include time bounds only when TRAIN_FROM/TRAIN_TO are defined
try:
    train_args = _base_args(from_time=TRAIN_FROM, to_time=TRAIN_TO)
    print(f"Training window : {TRAIN_FROM.isoformat()} → {TRAIN_TO.isoformat()}")
except NameError:
    train_args = _base_args()
    print(f"Training window : most recent {num_snapshots} snapshots (no fixed window)")

output_dir = Path("notebook_artifacts")
output_dir.mkdir(exist_ok=True)
snapshots_dir      = output_dir / "snapshots"
test_snapshots_dir = output_dir / "test_snapshots"

## 1. Ingest Training Data

Fetch network topology and metrics for the **training window** (`TRAIN_FROM` → `TRAIN_TO`) directly from your Spanner database using the SCD Type 2 logic inside `SpannerDataset`.

In [ ]:
print("Fetching training data from Spanner...")
snapshots = run_pipeline_local.step_ingest(train_args, snapshots_dir)
print(f"Ingested {len(snapshots)} training snapshots.")

## 1b. Ingest Test Data (optional)

Fetch **held-out test snapshots** from a separate time window (`TEST_FROM` → `TEST_TO`). These are saved separately and can be used to evaluate model performance on unseen data.

This cell is a no-op unless `TEST_FROM` and `TEST_TO` are defined in the Configuration cell above. Uncomment and set them to enable a fixed held-out split.

In [ ]:
try:
    TEST_FROM
    TEST_TO
except NameError:
    print("⚠  TEST_FROM / TEST_TO not defined — skipping test ingest.")
    print("   Uncomment and set them in the Configuration cell to fetch a held-out window.")
    test_snapshots = []
else:
    test_args = _base_args(from_time=TEST_FROM, to_time=TEST_TO)
    print(f"Fetching test data from Spanner ({TEST_FROM.isoformat()} → {TEST_TO.isoformat()})...")
    test_snapshots = run_pipeline_local.step_ingest(test_args, test_snapshots_dir)
    print(f"Ingested {len(test_snapshots)} test snapshots.")

## 2. Train HetGNN

Pass the fetched snapshots into the training script. This step:
1. Fits the `StandardScaler` on numerical features.
2. Converts data into PyTorch `HeteroData` graphs.
3. Trains the autoencoder over the specified number of epochs.
4. Saves `model.pth`, `scalers.pkl`, and `model_stats.pth` into our local `notebook_artifacts` folder.

In [ ]:
print("Training HetGNN on downloaded snapshots...")
model, val_loss, gb = train_hetgnn.train_hetgnn_on_snapshots(
    snapshot_objects=snapshots,
    output_dir=str(output_dir),
    hidden_channels=64, # Matches production defaults
    num_layers=2,
    epochs_override=50  # Reduced for rapid feedback (default is 50)
)
print(f"Training completed. Validation Loss: {val_loss:.4f}")

## 3. Serve (Inference)

Test the production serving code.
In production, `serve.py` queries Google Cloud Storage to find the latest trained model (via `latest_run.json`). To test it locally, we temporarily patch those loader functions to point to the artifacts we just trained in our `notebook_artifacts` directory.

This execution will:
1. Reload the model and scalers we just saved.
2. Fetch one *new* real snapshot from Spanner.
3. Perform a forward pass through the model to compute node embeddings and MSE reconstruction error.
4. **Write the results** directly back to the Spanner `NodeEmbedding` table.
5. Print the anomaly scores and a five-layer RCA decision tree.

In [ ]:
print("Setting up serving components with local artifacts instead of GCS...")

def mock_load_manifest():
    return {"run_id": "notebook_test_run"}

def mock_download_artefacts(manifest):
    return output_dir

async def test_serving():
    # Clear the in-process model cache so every run of this cell reloads the
    # freshly-trained artifacts from notebook_artifacts/.  Without this, the
    # cache key "notebook_test_run" would match a stale model from a previous
    # kernel execution and _ensure_model_loaded() would return early, silently
    # ignoring any retrained weights.
    serve._cache.clear()

    # We use unittest.mock.patch so serve.py doesn't try to download from GCS.
    # INTERVAL_MINUTES is patched to match the training cadence (0.5 min = 30 s)
    # so that rx_err_gradient is normalised by the same interval_seconds=30 that
    # was used during training — keeping the feature on the same scale the model
    # learned.  Without this patch the default (5 min) would make gradients 10x
    # smaller than the training distribution.
    with patch("serve._load_manifest", side_effect=mock_load_manifest), \
         patch("serve._download_artefacts", side_effect=mock_download_artefacts), \
         patch("serve.SPANNER_INSTANCE", spanner_instance), \
         patch("serve.SPANNER_DATABASE", spanner_database), \
         patch("serve.GOOGLE_CLOUD_PROJECT", project_id), \
         patch("serve.INTERVAL_MINUTES", interval_minutes):

        # Run the actual inference code
        results = await serve._run_inference()

        print("\n─── INFERENCE RESULTS ─────────────────────────────────────────")
        print(f"Snapshot timestamp : {results['snapshot_timestamp']}")
        print(f"Total anomalies    : {results['anomaly_count']}")

        # Per-type anomaly summary with per-type threshold
        thresholds = serve._cache.get("thresholds", serve._FALLBACK_THRESHOLDS)
        print("\n  Node Type        Nodes   Anomalies   Threshold")
        print("  " + "-"*52)
        for ntype, scores in results['anomaly_scores'].items():
            thresh = thresholds.get(ntype, 0.20)
            n_anom = sum(1 for s in scores if s > thresh)
            print(f"  {ntype:<18} {len(scores):>5}   {n_anom:>9}   MSE>{thresh:.2f}")

        # ── Five-layer RCA decision tree ──────────────────────────────────────
        # Covers all node types in the four-layer HetGNN model:
        #   router (control plane)  → interface (data plane link health)
        #   bgp_session (protocol)  → vrf (L3VPN policy)  → flow (end-to-end SLA)
        print("\n── RCA Decision Tree ────────────────────────────────────────────")

        r_scores = results['anomaly_scores'].get('router',      [])
        i_scores = results['anomaly_scores'].get('interface',   [])
        b_scores = results['anomaly_scores'].get('bgp_session', [])
        v_scores = results['anomaly_scores'].get('vrf',         [])
        f_scores = results['anomaly_scores'].get('flow',        [])

        r_expls = results.get('anomaly_explanations', {}).get('router',      [])
        i_expls = results.get('anomaly_explanations', {}).get('interface',   [])
        b_expls = results.get('anomaly_explanations', {}).get('bgp_session', [])
        v_expls = results.get('anomaly_explanations', {}).get('vrf',         [])
        f_expls = results.get('anomaly_explanations', {}).get('flow',        [])

        r_thresh = thresholds.get('router',      0.15)
        i_thresh = thresholds.get('interface',   0.20)
        b_thresh = thresholds.get('bgp_session', 0.10)
        v_thresh = thresholds.get('vrf',         0.10)
        f_thresh = thresholds.get('flow',        0.15)

        anom_routers    = [(i, s) for i, s in enumerate(r_scores) if s > r_thresh]
        anom_interfaces = [(i, s) for i, s in enumerate(i_scores) if s > i_thresh]
        anom_bgp        = [(i, s) for i, s in enumerate(b_scores) if s > b_thresh]
        anom_vrfs       = [(i, s) for i, s in enumerate(v_scores) if s > v_thresh]
        anom_flows      = [(i, s) for i, s in enumerate(f_scores) if s > f_thresh]

        def _top_feat_str(expl_list, idx):
            expl = expl_list[idx] if idx < len(expl_list) else {}
            top = expl.get('top_features', [])
            return ', '.join(f['feature'] for f in top) if top else 'n/a'

        def _top_feat_set(expl_list, idx):
            expl = expl_list[idx] if idx < len(expl_list) else {}
            return {f['feature'] for f in expl.get('top_features', [])}

        if not any([anom_routers, anom_interfaces, anom_bgp, anom_vrfs, anom_flows]):
            print("  ✓ All node types within normal MSE bounds — network looks healthy.")

        # ── 1. Router (control-plane health) ──────────────────────────────────
        if anom_routers:
            print(f"\n  [ROUTER] {len(anom_routers)} anomalous router(s) (MSE > {r_thresh}):")
            for idx, score in anom_routers[:5]:
                feats = _top_feat_set(r_expls, idx)
                print(f"    ▶ Router[{idx}] MSE={score:.4f}  top: {_top_feat_str(r_expls, idx)}")

                # Correlate: BGP sessions
                if anom_bgp:
                    print(f"      ↳ {len(anom_bgp)} BGP session(s) also anomalous:")
                    for bi, bs in anom_bgp[:3]:
                        print(f"         BGPSession[{bi}] MSE={bs:.4f}  top: {_top_feat_str(b_expls, bi)}")
                    b_feat_all = set().union(*[_top_feat_set(b_expls, bi) for bi, _ in anom_bgp])
                    if 'bgp_state' in b_feat_all or 'session_uptime_norm' in b_feat_all:
                        print("         → Likely BGP session flap / peer unreachable")
                    elif 'prefix_count_delta' in b_feat_all:
                        print("         → Likely prefix withdraw storm (route instability)")
                else:
                    if feats & {'cpu', 'mem'}:
                        print("      ↳ BGP healthy — likely CPU/memory resource exhaustion")
                    elif 'bgp_update_rate' in feats:
                        print("      ↳ Elevated BGP update rate — possible upstream route flap")
                    elif feats & {'fib_size_norm', 'ospf_num_routes'}:
                        print("      ↳ FIB/OSPF route count anomaly — possible routing table corruption")
                    else:
                        print("      ↳ BGP sessions healthy — check interfaces for link-layer fault")

                # Correlate: VRFs
                if anom_vrfs:
                    print(f"      ↳ {len(anom_vrfs)} VRF(s) also anomalous:")
                    for vi, vs in anom_vrfs[:3]:
                        vf = _top_feat_set(v_expls, vi)
                        if vf & {'rt_import_hash', 'rt_export_hash'}:
                            hint = "RT policy change — possible Fault 4/11 (import/export RT mismatch)"
                        elif 'vrf_route_count_delta' in vf:
                            hint = "VRF route churn — withdrawal or re-advertisement storm"
                        elif 'vrf_active_sessions' in vf:
                            hint = "Active BGP session count changed — session loss in L3VPN"
                        else:
                            hint = _top_feat_str(v_expls, vi)
                        print(f"         VRF[{vi}] MSE={vs:.4f}  → {hint}")

        # ── 2. Interface (data-plane / link health) ───────────────────────────
        if anom_interfaces:
            print(f"\n  [INTERFACE] {len(anom_interfaces)} anomalous interface(s) (MSE > {i_thresh}):")
            for idx, score in anom_interfaces[:5]:
                feats = _top_feat_set(i_expls, idx)
                print(f"    ▶ Interface[{idx}] MSE={score:.4f}  top: {_top_feat_str(i_expls, idx)}")
                if 'state' in feats:
                    print("      ↳ Interface down — check physical link / optics / SFP")
                elif feats & {'rx_errs_rate', 'rx_err_gradient'}:
                    print("      ↳ Rising receive errors — hardware fault or bad cable (Fault 3)")
                elif 'tx_queue_len_norm' in feats:
                    print("      ↳ TX queue depth elevated — egress congestion or policer (Fault 8)")
                elif feats & {'tx_util', 'rx_util'}:
                    print("      ↳ High utilisation — bandwidth saturation or traffic surge")
                elif feats & {'rx_drops', 'tx_drops'}:
                    print("      ↳ Packet drops elevated — possible buffer overflow or rate-limiting")

        # ── 3. BGP session (standalone — not already shown under router) ──────
        if anom_bgp and not anom_routers:
            print(f"\n  [BGP SESSION] {len(anom_bgp)} anomalous session(s) (MSE > {b_thresh}):")
            for idx, score in anom_bgp[:5]:
                feats = _top_feat_set(b_expls, idx)
                print(f"    ▶ BGPSession[{idx}] MSE={score:.4f}  top: {_top_feat_str(b_expls, idx)}")
                if 'bgp_state' in feats or 'session_uptime_norm' in feats:
                    print("      ↳ Session down or recently flapped — check peer reachability")
                elif 'prefix_count_delta' in feats:
                    print("      ↳ Prefix count changing rapidly — route withdraw/re-announce storm")
                elif 'pfx_count_norm' in feats:
                    print("      ↳ Prefix count at unusual level — possible route leak or filtering")

        # ── 4. VRF (L3VPN policy / routing) — standalone ──────────────────────
        if anom_vrfs and not anom_routers:
            print(f"\n  [VRF] {len(anom_vrfs)} anomalous VRF(s) (MSE > {v_thresh}):")
            for idx, score in anom_vrfs[:5]:
                feats = _top_feat_set(v_expls, idx)
                print(f"    ▶ VRF[{idx}] MSE={score:.4f}  top: {_top_feat_str(v_expls, idx)}")
                if feats & {'rt_import_hash', 'rt_export_hash'}:
                    print("      ↳ RT import/export hash changed — RT policy misconfiguration (Fault 4/11)")
                elif 'vrf_route_count_delta' in feats:
                    print("      ↳ VRF route count delta elevated — route churn / withdrawal")
                elif 'vrf_active_sessions' in feats:
                    print("      ↳ Active BGP sessions in VRF changed — session loss in L3VPN")

        # ── 5. Flow (end-to-end SLA) ──────────────────────────────────────────
        if anom_flows:
            print(f"\n  [FLOW] {len(anom_flows)} anomalous flow(s) (MSE > {f_thresh}):")
            for idx, score in anom_flows[:5]:
                feats = _top_feat_set(f_expls, idx)
                print(f"    ▶ Flow[{idx}] MSE={score:.4f}  top: {_top_feat_str(f_expls, idx)}")
                if 'packet_loss_pct' in feats:
                    print("      ↳ Packet loss detected — link fault, queue overflow, or policing")
                elif feats & {'latency_ms_norm', 'jitter_norm'}:
                    print("      ↳ Latency/jitter spike — possible path change, congestion, or queue buildup")
                elif feats & {'throughput_bps', 'throughput_delta'}:
                    print("      ↳ Throughput drop — traffic disruption, rate-limiting, or path failure")
                elif 'active_sessions' in feats:
                    print("      ↳ Active session count anomaly — application or connectivity issue")

# Await the async server inference method
await test_serving()
